DATA EXPLORATION - 1 

In [ ]:
import requests

# Leggi l'indice completo dei file disponibili
r = requests.get("https://api.fda.gov/download.json")
data = r.json()

# Estrai i file degli adverse events
partitions = data['results']['drug']['event']['partitions']

print(f"Numero totale di file: {len(partitions)}")
print("\nPrimi 5 file disponibili:")
for p in partitions[:5]:
    print(f"  {p['file']}")
    print(f"  Records: {p.get('records', 'N/A')}")
    print()

scarico il primo file per capire la struttura 

In [ ]:
from pathlib import Path
import requests

Path("../data/raw").mkdir(parents=True, exist_ok=True)

url = partitions[0]['file']
filename = url.split('/')[-1]
dest = f"../data/raw/{filename}"

print(f"Scaricando: {filename}")
r = requests.get(url, stream=True)
total = int(r.headers.get('content-length', 0))
downloaded = 0

with open(dest, 'wb') as f:
    for chunk in r.iter_content(chunk_size=8192):
        f.write(chunk)
        downloaded += len(chunk)
        pct = downloaded / total * 100 if total else 0
        print(f"\r{pct:.1f}% — {downloaded/1e6:.1f} MB", end='')

print(f"\nCompletato: {dest}")

In [ ]:
import zipfile
import ijson
import json

zip_path = dest

with zipfile.ZipFile(zip_path) as zf:
    json_filename = [f for f in zf.namelist() if f.endswith('.json')][0]
    print(f"File dentro lo zip: {json_filename}\n")
    
    with zf.open(json_filename) as f:
        # Leggi solo il primo record
        records = ijson.items(f, 'results.item')
        first = next(records)

# Stampa il record completo formattato
print(json.dumps(first, indent=2))

c'è qualche incongruenza con la struttura descritta su openFDA ma di base ci siamo. 
- il nome del farmaco è standardizzato in maniera strana
- manca "reaction outcome"
- manca "active substance" che è un problema (non ce il principio attivo) 

In [ ]:
with zipfile.ZipFile(zip_path) as zf:
    json_filename = [f for f in zf.namelist() if f.endswith('.json')][0]
    with zf.open(json_filename) as f:
        records = ijson.items(f, 'results.item')
        sample = [next(records) for _ in range(20)]

# Analisi rapida sui 20 record
print("=== CAMPI PRESENTI ===")
all_keys = set()
for r in sample:
    all_keys.update(r.keys())
print("Top level:", sorted(all_keys))

print("\n=== CAMPI PATIENT ===")
patient_keys = set()
for r in sample:
    patient_keys.update(r.get('patient', {}).keys())
print("Patient:", sorted(patient_keys))

print("\n=== CAMPI DRUG ===")
drug_keys = set()
for r in sample:
    for d in r.get('patient', {}).get('drug', []):
        drug_keys.update(d.keys())
print("Drug:", sorted(drug_keys))

print("\n=== CAMPI REACTION ===")
rxn_keys = set()
for r in sample:
    for rx in r.get('patient', {}).get('reaction', []):
        rxn_keys.update(rx.keys())
print("Reaction:", sorted(rxn_keys))

print("\n=== STATISTICHE ===")
print(f"Record con activesubstance: "
      f"{sum(1 for r in sample for d in r.get('patient',{}).get('drug',[]) if 'activesubstance' in d)}"
      f" drug su {sum(len(r.get('patient',{}).get('drug',[])) for r in sample)} totali")

print(f"Record con reactionoutcome: "
      f"{sum(1 for r in sample for rx in r.get('patient',{}).get('reaction',[]) if 'reactionoutcome' in rx)}"
      f" reazioni su {sum(len(r.get('patient',{}).get('reaction',[])) for r in sample)} totali")

print(f"Record con patientagegroup: "
      f"{sum(1 for r in sample if r.get('patient',{}).get('patientagegroup'))}"
      f" su {len(sample)} totali")

ce campo OPEN FDA che non è segnato nella documentazione 

In [ ]:
print("=== CAMPO OPENFDA ===")
for r in sample:
    for d in r.get('patient', {}).get('drug', []):
        if 'openfda' in d and d['openfda']:
            print(json.dumps(d['openfda'], indent=2))
            break
    else:
        continue
    break

TOP! OPENFDA risolve buona parte dei problemi. mo capiamo quanto è cambiata la struttura di sti dati negli anni. 

In [ ]:
# Trova l'ultimo file disponibile (il più recente)
last_partition = partitions[-1]
print(f"File più recente: {last_partition['file']}")
print(f"Records: {last_partition.get('records')}")

url_recent = last_partition['file']
filename_recent = url_recent.split('/')[-1]
dest_recent = f"../data/raw/{filename_recent}"

print(f"\nScaricando file recente: {filename_recent}")
r = requests.get(url_recent, stream=True)
total = int(r.headers.get('content-length', 0))
downloaded = 0

with open(dest_recent, 'wb') as f:
    for chunk in r.iter_content(chunk_size=8192):
        f.write(chunk)
        downloaded += len(chunk)
        pct = downloaded / total * 100 if total else 0
        print(f"\r{pct:.1f}% — {downloaded/1e6:.1f} MB", end='')

print(f"\nCompletato: {dest_recent}")

In [ ]:
with zipfile.ZipFile(dest_recent) as zf:
    json_filename = [f for f in zf.namelist() if f.endswith('.json')][0]
    with zf.open(json_filename) as f:
        records = ijson.items(f, 'results.item')
        sample_recent = [next(records) for _ in range(20)]

print("=== CAMPI DRUG (file recente) ===")
drug_keys_recent = set()
for r in sample_recent:
    for d in r.get('patient', {}).get('drug', []):
        drug_keys_recent.update(d.keys())
print("Drug:", sorted(drug_keys_recent))

print("\n=== CAMPI REACTION (file recente) ===")
rxn_keys_recent = set()
for r in sample_recent:
    for rx in r.get('patient', {}).get('reaction', []):
        rxn_keys_recent.update(rx.keys())
print("Reaction:", sorted(rxn_keys_recent))

print("\n=== STATISTICHE (file recente) ===")
print(f"Record con activesubstance: "
      f"{sum(1 for r in sample_recent for d in r.get('patient',{}).get('drug',[]) if 'activesubstance' in d)}"
      f" drug su {sum(len(r.get('patient',{}).get('drug',[])) for r in sample_recent)} totali")

print(f"Record con reactionoutcome: "
      f"{sum(1 for r in sample_recent for rx in r.get('patient',{}).get('reaction',[]) if 'reactionoutcome' in rx)}"
      f" reazioni su {sum(len(r.get('patient',{}).get('reaction',[])) for r in sample_recent)} totali")

print(f"Record con patientagegroup: "
      f"{sum(1 for r in sample_recent if r.get('patient',{}).get('patientagegroup'))}"
      f" su {len(sample_recent)} totali")

# Mostra un record completo recente
print("\n=== PRIMO RECORD RECENTE ===")
print(json.dumps(sample_recent[0], indent=2))

In [ ]:
print("=== CAMPO OPENFDA ===")
for r in sample_recent:
    for d in r.get('patient', {}).get('drug', []):
        if 'openfda' in d and d['openfda']:
            print(json.dumps(d['openfda'], indent=2))
            break
    else:
        continue
    break

ATTENZIONE perche la chiamata con API non restituisce il file piu recente effettivamente (non sono indicizzate). vedi cella dopo

In [ ]:
# Trova il file più recente per data nell'URL
def extract_year_quarter(url):
    match = re.search(r'/(\d{4})q(\d)/', url)
    if match:
        return int(match.group(1)), int(match.group(2))
    return (0, 0)

# Ordina tutti i file per data
partitions_sorted = sorted(partitions, key=lambda p: extract_year_quarter(p['file']))

print(f"Primo file (più vecchio): {partitions_sorted[0]['file']}")
print(f"Ultimo file (più recente): {partitions_sorted[-1]['file']}")

print("\nUltimi 5 file disponibili:")
for p in partitions_sorted[-5:]:
    print(f"  {p['file']}")
    print(f"  Records: {p.get('records', 'N/A')}")

In [ ]:
url_recent = partitions_sorted[-1]['file']
filename_recent = url_recent.split('/')[-1]
dest_recent = f"../data/raw/{filename_recent}"

print(f"Scaricando file più recente: {filename_recent}")
r = requests.get(url_recent, stream=True)
total = int(r.headers.get('content-length', 0))
downloaded = 0

with open(dest_recent, 'wb') as f:
    for chunk in r.iter_content(chunk_size=8192):
        f.write(chunk)
        downloaded += len(chunk)
        pct = downloaded / total * 100 if total else 0
        print(f"\r{pct:.1f}% — {downloaded/1e6:.1f} MB", end='')

print(f"\nCompletato: {dest_recent}")

In [ ]:
# Primo record del file recente
with zipfile.ZipFile(dest_recent) as zf:
    json_filename = [f for f in zf.namelist() if f.endswith('.json')][0]
    print(f"File dentro lo zip: {json_filename}\n")
    with zf.open(json_filename) as f:
        records = ijson.items(f, 'results.item')
        sample_2025 = [next(records) for _ in range(20)]

print(json.dumps(sample_2025[0], indent=2))

OK guardando la struttura generale dell'ultimo farmaco è tutto ok. 
confermato che _generic_name_ non è affidabile, ma solo un fallback per identificare il farmaco. la prima scelta rimane _activesubstance.activesubstance_name_. 
c'è reaction outcome, e abbiamo anche _seriousnesslifethreatening_, _seriousnesshospitalization_